# LeetCode 1122 — Relative Sort Array bằng Python

**Link đề:** https://leetcode.com/problems/relative-sort-array/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting, Counting Sort

```python
class Solution:
    def relativeSortArray(self, arr1: List[int], arr2: List[int]) -> List[int]:
        ...
```

## Mục tiêu học
1. Hiểu bài toán **sắp xếp theo thứ tự tùy chỉnh** (custom order): thứ tự do `arr2` quy định, phần còn lại tăng dần.
2. Thành thạo `sorted(..., key=...)`, `lambda`, và mẹo **so sánh tuple** để sắp xếp theo nhiều tiêu chí.
3. Dùng `dict` ánh xạ giá trị → thứ hạng (rank) để tra cứu `O(1)` trung bình.
4. Học **counting sort** (sắp xếp đếm) tận dụng miền giá trị nhỏ `0..1000` để đạt `O(n + V)`.
5. Trace, chứng minh tính đúng, Time/Space, bộ test đối chiếu, biến thể (#791 Custom Sort String) và bài tập.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — sắp xếp `arr1` theo thứ tự của `arr2`

Cho hai mảng `arr1` và `arr2`; các phần tử của `arr2` **khác nhau đôi một** và **đều có mặt** trong `arr1`.

Sắp xếp `arr1` sao cho:
1. Các phần tử có trong `arr2` xuất hiện theo **đúng thứ tự tương đối** như trong `arr2` (giá trị lặp thì đứng liền nhau).
2. Các phần tử **không** có trong `arr2` đặt ở **cuối**, theo thứ tự **tăng dần**.

| arr1 | arr2 | Output |
|---|---|---|
| `[2,3,1,3,2,4,6,7,9,2,19]` | `[2,1,4,3,9,6]` | `[2,2,2,1,4,3,3,9,6,7,19]` |
| `[28,6,22,8,44,17]` | `[22,28,8,6]` | `[22,28,8,6,17,44]` |

**Constraints:** `1 <= arr1.length, arr2.length <= 1000`; `0 <= arr1[i], arr2[i] <= 1000`; phần tử `arr2` phân biệt; mỗi `arr2[i]` có trong `arr1`.

**Gợi ý của đề:** dùng hashmap ánh xạ giá trị của `arr2` sang vị trí của nó, rồi dùng hàm sắp xếp tùy chỉnh.

In [1]:
arr1 = [2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19]
arr2 = [2, 1, 4, 3, 9, 6]
in_arr2 = [x for x in arr1 if x in arr2]
not_in = [x for x in arr1 if x not in arr2]
print("phần tử có trong arr2   :", in_arr2)
print("phần tử KHÔNG có trong arr2:", not_in, "-> tăng dần:", sorted(not_in))

phần tử có trong arr2   : [2, 3, 1, 3, 2, 4, 6, 9, 2]
phần tử KHÔNG có trong arr2: [7, 19] -> tăng dần: [7, 19]


## 2. Nền tảng Python

### 2.1. `sorted(iterable, key=...)` — sắp xếp theo "khóa"

`sorted` không so sánh trực tiếp phần tử mà so sánh `key(phần_tử)`. Ví dụ sắp xếp chuỗi theo độ dài: `key=len`.

**Sắp xếp ổn định (stable):** các phần tử có khóa bằng nhau giữ nguyên thứ tự ban đầu. Python luôn sắp xếp ổn định.

In [2]:
words = ["banana", "kiwi", "apple", "fig", "cherry"]
print(sorted(words))                 # theo bảng chữ cái
print(sorted(words, key=len))        # theo độ dài; 'banana','cherry' cùng 6 -> giữ thứ tự gốc
print(sorted([3, 1, 2], reverse=True))

['apple', 'banana', 'cherry', 'fig', 'kiwi']
['fig', 'kiwi', 'apple', 'banana', 'cherry']
[3, 2, 1]


### 2.2. `lambda` — hàm vô danh một dòng

`lambda x: x % 10` tương đương:

```python
def f(x):
    return x % 10
```

Hay dùng làm `key` cho `sorted`.

In [3]:
nums = [25, 13, 41, 7, 30]
print(sorted(nums, key=lambda x: x % 10))     # theo chữ số hàng đơn vị
f = lambda x: x % 10
print(f(25), f(13))

[30, 41, 13, 25, 7]
5 3


### 2.3. So sánh tuple — sắp xếp theo nhiều tiêu chí

Python so sánh tuple **từ trái sang phải**: so phần tử đầu; nếu bằng nhau mới so phần tử thứ hai, …

`(0, 5) < (1, 0)` vì `0 < 1` (không cần xét phần tử thứ hai). `(1, 3) < (1, 7)` vì phần tử đầu bằng nhau, `3 < 7`.

Mẹo: khóa `(nhóm, giá trị trong nhóm)` — nhóm nhỏ đứng trước, trong nhóm sắp theo giá trị.

In [4]:
print((0, 5) < (1, 0), (1, 3) < (1, 7), (2, 1) < (1, 9))
pairs = [(1, 7), (0, 5), (1, 3), (0, 2)]
print(sorted(pairs))

True True False
[(0, 2), (0, 5), (1, 3), (1, 7)]


### 2.4. `dict` ánh xạ giá trị → vị trí (rank)

`{v: i for i, v in enumerate(arr2)}` là **dict comprehension**: tạo dict mà key là giá trị, value là chỉ số trong `arr2`. Tra `rank[v]` trung bình `O(1)`; kiểm tra `v in rank` cũng `O(1)` trung bình.

In [5]:
arr2 = [2, 1, 4, 3, 9, 6]
rank = {v: i for i, v in enumerate(arr2)}
print(rank)
print("rank[4] =", rank[4], "| 7 in rank?", 7 in rank, "| rank.get(7, 'không có') =", rank.get(7, "không có"))

{2: 0, 1: 1, 4: 2, 3: 3, 9: 4, 6: 5}
rank[4] = 2 | 7 in rank? False | rank.get(7, 'không có') = không có


## 3. Cách 1 — Trực giác: gom từng giá trị của `arr2`, phần dư sắp riêng

Với mỗi `v` trong `arr2` (đúng thứ tự), lấy mọi phần tử bằng `v` trong `arr1`. Sau đó lấy các phần tử không thuộc `arr2`, sắp tăng dần, nối vào cuối.

In [6]:
from typing import List

def relative_sort_naive(arr1: List[int], arr2: List[int]) -> List[int]:
    result = []
    for v in arr2:
        for x in arr1:            # quét cả arr1 cho mỗi v
            if x == v:
                result.append(x)
    rest = [x for x in arr1 if x not in arr2]     # 'in' trên list: O(m)
    return result + sorted(rest)

print(relative_sort_naive([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6]))
print(relative_sort_naive([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))

[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]
[22, 28, 8, 6, 17, 44]


### Nút thắt

Gọi `n = len(arr1)`, `m = len(arr2)`:
- Vòng lồng `for v in arr2: for x in arr1` → `O(n·m)`.
- `x not in arr2` với `arr2` là **list** → dò tuần tự `O(m)` cho mỗi `x` → thêm `O(n·m)`.

Với `n = m = 1000` là ~`2·10^6` phép so sánh — vẫn qua, nhưng chậm không cần thiết. Hai chỗ lặp đi lặp lại: "tìm `v` trong `arr1`" và "hỏi `x` có thuộc `arr2`". Cả hai đều giải được bằng **bảng băm** hoặc **đếm**.

## 4. Insight — biến "thứ tự tùy chỉnh" thành một **khóa sắp xếp**

Gán cho mỗi phần tử `x` một khóa:
- Nếu `x` thuộc `arr2`: khóa `(0, rank[x])` — nhóm 0, xếp theo vị trí trong `arr2`.
- Nếu không: khóa `(1, x)` — nhóm 1 (luôn sau nhóm 0), xếp theo giá trị.

Rồi chỉ việc `sorted(arr1, key=...)`. Mọi quy tắc của đề được mã hóa trong khóa.

In [7]:
arr1 = [2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19]
arr2 = [2, 1, 4, 3, 9, 6]
rank = {v: i for i, v in enumerate(arr2)}

def key_of(x):
    return (0, rank[x]) if x in rank else (1, x)

for x in [2, 3, 7, 19, 6]:
    print(x, "-> khóa", key_of(x))

2 -> khóa (0, 0)
3 -> khóa (0, 3)
7 -> khóa (1, 7)
19 -> khóa (1, 19)
6 -> khóa (0, 5)


## 5. Cách 2 — `sorted` với khóa tuple (lời giải chính, theo gợi ý)

In [8]:
def relative_sort_key(arr1: List[int], arr2: List[int]) -> List[int]:
    rank = {v: i for i, v in enumerate(arr2)}
    return sorted(arr1, key=lambda x: (0, rank[x]) if x in rank else (1, x))

print(relative_sort_key([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6]))
print(relative_sort_key([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))

[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]
[22, 28, 8, 6, 17, 44]


Một mẹo khóa khác chỉ dùng **một số**: phần tử thuộc `arr2` có khóa `rank[x]` (từ `0` đến `m-1`), phần tử ngoài có khóa `m + x` (luôn `>= m`). Vì `x >= 0`, nhóm ngoài luôn đứng sau.

In [9]:
def relative_sort_key_int(arr1: List[int], arr2: List[int]) -> List[int]:
    rank = {v: i for i, v in enumerate(arr2)}
    m = len(arr2)
    return sorted(arr1, key=lambda x: rank.get(x, m + x))

print(relative_sort_key_int([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6]))

[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]


## 6. TRACE — khóa của từng phần tử và kết quả sắp xếp

Ví dụ 2: `arr1 = [28,6,22,8,44,17]`, `arr2 = [22,28,8,6]` → `rank = {22:0, 28:1, 8:2, 6:3}`.

| x | thuộc arr2? | khóa | |
|---:|:---:|---|---|
| 28 | Có | `(0, 1)` | |
| 6 | Có | `(0, 3)` | |
| 22 | Có | `(0, 0)` | nhỏ nhất → đứng đầu |
| 8 | Có | `(0, 2)` | |
| 44 | Không | `(1, 44)` | |
| 17 | Không | `(1, 17)` | |

Sắp theo khóa: `(0,0)=22, (0,1)=28, (0,2)=8, (0,3)=6, (1,17)=17, (1,44)=44` → `[22,28,8,6,17,44]`.

In [10]:
def trace_relative_sort(arr1, arr2):
    rank = {v: i for i, v in enumerate(arr2)}
    print("rank =", rank)
    keyed = []
    for x in arr1:
        k = (0, rank[x]) if x in rank else (1, x)
        keyed.append((k, x))
        print(f"x={x:>3} | thuộc arr2? {str(x in rank):<5} | khóa={k}")
    keyed.sort()                         # sort theo khóa (tuple so sánh từ trái sang)
    print("Sau khi sắp theo khóa:")
    for k, x in keyed:
        print("  ", k, "->", x)
    result = [x for _, x in keyed]
    print("Kết quả:", result)
    return result

_ = trace_relative_sort([28, 6, 22, 8, 44, 17], [22, 28, 8, 6])

rank = {22: 0, 28: 1, 8: 2, 6: 3}
x= 28 | thuộc arr2? True  | khóa=(0, 1)
x=  6 | thuộc arr2? True  | khóa=(0, 3)
x= 22 | thuộc arr2? True  | khóa=(0, 0)
x=  8 | thuộc arr2? True  | khóa=(0, 2)
x= 44 | thuộc arr2? False | khóa=(1, 44)
x= 17 | thuộc arr2? False | khóa=(1, 17)
Sau khi sắp theo khóa:
   (0, 0) -> 22
   (0, 1) -> 28
   (0, 2) -> 8
   (0, 3) -> 6
   (1, 17) -> 17
   (1, 44) -> 44
Kết quả: [22, 28, 8, 6, 17, 44]


Trong `[x for _, x in keyed]`, dấu `_` là tên biến quy ước cho "giá trị không dùng tới".

## 7. Cách 3 — Counting Sort (tận dụng `0 <= giá trị <= 1000`)

Ý tưởng **sắp xếp đếm**:
1. Đếm `count[v]` = số lần `v` xuất hiện trong `arr1` (mảng 1001 ô).
2. Duyệt `arr2` theo thứ tự: với mỗi `v`, thêm `v` đúng `count[v]` lần, rồi đặt `count[v] = 0`.
3. Duyệt `v` từ `0` đến `1000` (tăng dần!): thêm `v` đúng `count[v]` lần — đó là các phần tử còn lại, tự động tăng dần.

Không cần so sánh nào → `O(n + m + V)` với `V = 1001`.

In [11]:
# Minh họa bước 1: đếm
arr1 = [2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19]
count = [0] * 20                  # đủ cho ví dụ nhỏ (giá trị < 20)
for x in arr1:
    count[x] += 1
print({v: c for v, c in enumerate(count) if c > 0})
print("[2] * 3 =", [2] * 3, "| list.extend thêm nhiều phần tử một lúc")

{1: 1, 2: 3, 3: 2, 4: 1, 6: 1, 7: 1, 9: 1, 19: 1}
[2] * 3 = [2, 2, 2] | list.extend thêm nhiều phần tử một lúc


In [12]:
def relative_sort_counting(arr1: List[int], arr2: List[int]) -> List[int]:
    MAXV = 1000
    count = [0] * (MAXV + 1)
    for x in arr1:
        count[x] += 1
    result = []
    for v in arr2:                      # phần theo thứ tự arr2
        result.extend([v] * count[v])
        count[v] = 0                    # đã dùng hết
    for v in range(MAXV + 1):           # phần còn lại, tăng dần
        if count[v]:
            result.extend([v] * count[v])
    return result

print(relative_sort_counting([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6]))
print(relative_sort_counting([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))

[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]
[22, 28, 8, 6, 17, 44]


### 7.1. Trace counting sort

Ví dụ 1: sau khi đếm, `count = {1:1, 2:3, 3:2, 4:1, 6:1, 7:1, 9:1, 19:1}`.

| Giai đoạn | v | count[v] | thêm | result |
|---|---:|---:|---|---|
| arr2 | 2 | 3 | `2,2,2` | `[2,2,2]` |
| arr2 | 1 | 1 | `1` | `[2,2,2,1]` |
| arr2 | 4 | 1 | `4` | `[...,4]` |
| arr2 | 3 | 2 | `3,3` | `[...,3,3]` |
| arr2 | 9 | 1 | `9` | `[...,9]` |
| arr2 | 6 | 1 | `6` | `[...,6]` |
| còn lại | 7 | 1 | `7` | `[...,7]` |
| còn lại | 19 | 1 | `19` | `[...,19]` |

In [13]:
def trace_counting(arr1, arr2, maxv=20):
    count = [0] * (maxv + 1)
    for x in arr1:
        count[x] += 1
    print("count (khác 0):", {v: c for v, c in enumerate(count) if c})
    result = []
    for v in arr2:
        result.extend([v] * count[v])
        print(f"arr2: v={v:>2}, count={count[v]} -> result={result}")
        count[v] = 0
    for v in range(maxv + 1):
        if count[v]:
            result.extend([v] * count[v])
            print(f"còn lại: v={v:>2}, count={count[v]} -> result={result}")
    return result

_ = trace_counting([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6])

count (khác 0): {1: 1, 2: 3, 3: 2, 4: 1, 6: 1, 7: 1, 9: 1, 19: 1}
arr2: v= 2, count=3 -> result=[2, 2, 2]
arr2: v= 1, count=1 -> result=[2, 2, 2, 1]
arr2: v= 4, count=1 -> result=[2, 2, 2, 1, 4]
arr2: v= 3, count=2 -> result=[2, 2, 2, 1, 4, 3, 3]
arr2: v= 9, count=1 -> result=[2, 2, 2, 1, 4, 3, 3, 9]
arr2: v= 6, count=1 -> result=[2, 2, 2, 1, 4, 3, 3, 9, 6]
còn lại: v= 7, count=1 -> result=[2, 2, 2, 1, 4, 3, 3, 9, 6, 7]
còn lại: v=19, count=1 -> result=[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]


## 8. Cách 4 — `collections.Counter` (đếm bằng dict, không phụ thuộc miền giá trị)

`Counter(arr1)` là dict đặc biệt: key là giá trị, value là số lần xuất hiện. `cnt.pop(v)` lấy ra và xóa key `v`. Các key còn lại sắp tăng dần bằng `sorted(cnt)`.

Cách này dùng khi giá trị có thể rất lớn/âm (không làm được mảng đếm).

In [14]:
from collections import Counter

c = Counter([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19])
print(c)
print("c[2] =", c[2], "| c[100] =", c[100], "(key không có -> 0, không lỗi)")

def relative_sort_counter(arr1: List[int], arr2: List[int]) -> List[int]:
    cnt = Counter(arr1)
    result = []
    for v in arr2:
        result.extend([v] * cnt.pop(v))
    for v in sorted(cnt):
        result.extend([v] * cnt[v])
    return result

print(relative_sort_counter([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6]))

Counter({2: 3, 3: 2, 1: 1, 4: 1, 6: 1, 7: 1, 9: 1, 19: 1})
c[2] = 3 | c[100] = 0 (key không có -> 0, không lỗi)
[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]


## 9. Code nộp LeetCode

Bản khóa tuple (ngắn, đúng gợi ý của đề):

In [15]:
from typing import List

class Solution:
    def relativeSortArray(self, arr1: List[int], arr2: List[int]) -> List[int]:
        rank = {v: i for i, v in enumerate(arr2)}
        return sorted(arr1, key=lambda x: (0, rank[x]) if x in rank else (1, x))

sol = Solution()
print(sol.relativeSortArray([2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6]))
print(sol.relativeSortArray([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))

[2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]
[22, 28, 8, 6, 17, 44]


Bản counting sort (tuyến tính):

In [16]:
class SolutionCounting:
    def relativeSortArray(self, arr1: List[int], arr2: List[int]) -> List[int]:
        count = [0] * 1001
        for x in arr1:
            count[x] += 1
        result = []
        for v in arr2:
            result.extend([v] * count[v])
            count[v] = 0
        for v in range(1001):
            result.extend([v] * count[v])
        return result

print(SolutionCounting().relativeSortArray([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))

[22, 28, 8, 6, 17, 44]


## 10. Vì sao đúng? — Độ phức tạp — So sánh

**Cách khóa tuple:** với hai phần tử `x, y`:
- cả hai thuộc `arr2`: khóa `(0, rank[x])` vs `(0, rank[y])` → so theo vị trí trong `arr2` — đúng yêu cầu 1; giá trị bằng nhau có khóa bằng nhau nên đứng liền nhau.
- một thuộc, một không: `(0, ·) < (1, ·)` → phần tử thuộc `arr2` đứng trước — đúng yêu cầu 2.
- cả hai không thuộc: `(1, x)` vs `(1, y)` → tăng dần theo giá trị.
Một thứ tự toàn phần trên khóa thỏa mọi cặp ⇒ kết quả sắp xếp đúng.

**Counting sort:** sau bước đếm, mỗi giá trị `v` xuất hiện trong `result` đúng `count[v]` lần (không mất, không thừa) vì ta đặt `count[v] = 0` sau khi dùng ở bước `arr2`. Bước `arr2` sinh các nhóm theo đúng thứ tự `arr2`; bước còn lại duyệt `v` tăng dần nên phần cuối tăng dần.

Gọi `n = len(arr1)`, `m = len(arr2)`, `V = 1001`, `k` = số giá trị phân biệt ngoài `arr2`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Trực giác | `O(n·m + n log n)` | `O(n)` |
| 2. Khóa tuple + dict | `O(n log n + m)` | `O(n + m)` |
| 3. Counting sort | `O(n + m + V)` | `O(V)` |
| 4. Counter | `O(n + m + k log k)` | `O(n)` |

## 11. Bộ test — mẫu, biên, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS {actual}")

cases = [
    ("mẫu 1", [2, 3, 1, 3, 2, 4, 6, 7, 9, 2, 19], [2, 1, 4, 3, 9, 6], [2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]),
    ("mẫu 2", [28, 6, 22, 8, 44, 17], [22, 28, 8, 6], [22, 28, 8, 6, 17, 44]),
    ("1 phần tử", [5], [5], [5]),
    ("arr2 chứa mọi giá trị", [3, 1, 3, 1], [3, 1], [3, 3, 1, 1]),
    ("phần ngoài có lặp", [9, 0, 5, 9, 0], [5], [5, 0, 0, 9, 9]),
    ("giá trị biên 0 và 1000", [1000, 0, 7, 1000], [7], [7, 0, 1000, 1000]),
    ("arr2 thứ tự giảm", [1, 2, 3, 4], [4, 3, 2, 1], [4, 3, 2, 1]),
]
for name, a1, a2, exp in cases:
    check(name, Solution().relativeSortArray(list(a1), list(a2)), exp)

mẫu 1                          PASS [2, 2, 2, 1, 4, 3, 3, 9, 6, 7, 19]
mẫu 2                          PASS [22, 28, 8, 6, 17, 44]
1 phần tử                      PASS [5]
arr2 chứa mọi giá trị          PASS [3, 3, 1, 1]
phần ngoài có lặp              PASS [5, 0, 0, 9, 9]
giá trị biên 0 và 1000         PASS [7, 0, 1000, 1000]
arr2 thứ tự giảm               PASS [4, 3, 2, 1]


In [18]:
import random
rng = random.Random(1122)
funcs = [relative_sort_naive, relative_sort_key, relative_sort_key_int, relative_sort_counting,
         relative_sort_counter, Solution().relativeSortArray, SolutionCounting().relativeSortArray]
for _ in range(1500):
    arr1 = [rng.randint(0, 30) for _ in range(rng.randint(1, 25))]
    distinct = list(set(arr1))
    rng.shuffle(distinct)
    arr2 = distinct[:rng.randint(1, len(distinct))]      # phân biệt và đều có trong arr1
    expected = relative_sort_naive(arr1, arr2)
    for f in funcs:
        assert f(list(arr1), list(arr2)) == expected, (f, arr1, arr2)
    assert sorted(expected) == sorted(arr1)               # là hoán vị của arr1
print("1500 bộ ngẫu nhiên: 7 cách khớp nhau, kết quả là hoán vị của arr1")

1500 bộ ngẫu nhiên: 7 cách khớp nhau, kết quả là hoán vị của arr1


## 12. Lỗi người mới hay gặp

### Lỗi 1: Khóa chỉ dùng `rank`, gán phần tử ngoài một hằng số

In [19]:
def wrong_constant_key(arr1, arr2):
    rank = {v: i for i, v in enumerate(arr2)}
    return sorted(arr1, key=lambda x: rank.get(x, len(arr2)))   # SAI: phần ngoài cùng khóa

print("sai :", wrong_constant_key([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))
print("đúng:", relative_sort_key([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))

sai : [22, 28, 8, 6, 44, 17]
đúng: [22, 28, 8, 6, 17, 44]


**Vì sao sai:** mọi phần tử ngoài `arr2` có cùng khóa → sắp xếp ổn định giữ **thứ tự gốc** (`44, 17`) chứ không tăng dần. **Sửa:** khóa `(1, x)` hoặc `m + x`.

### Lỗi 2: Khóa `m + x` khi giá trị có thể âm

In [20]:
def key_int_sort(arr1, arr2):
    rank = {v: i for i, v in enumerate(arr2)}
    return sorted(arr1, key=lambda x: rank.get(x, len(arr2) + x))

print("với giá trị âm (ngoài constraints):", key_int_sort([5, -10, 3], [3, 5]))
print("khóa tuple vẫn đúng            :", relative_sort_key([5, -10, 3], [3, 5]))

với giá trị âm (ngoài constraints): [-10, 3, 5]
khóa tuple vẫn đúng            : [3, 5, -10]


**Bài học:** mẹo `m + x` chỉ đúng vì đề bảo đảm `x >= 0`. Khóa tuple `(nhóm, giá trị)` an toàn trong mọi trường hợp.

### Lỗi 3: Counting sort quên đặt `count[v] = 0` sau bước `arr2`

In [21]:
def wrong_no_reset(arr1, arr2):
    count = [0] * 1001
    for x in arr1:
        count[x] += 1
    result = []
    for v in arr2:
        result.extend([v] * count[v])              # quên count[v] = 0
    for v in range(1001):
        result.extend([v] * count[v])
    return result

print("sai :", wrong_no_reset([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))

sai : [22, 28, 8, 6, 6, 8, 17, 22, 28, 44]


**Vì sao sai:** các giá trị của `arr2` bị thêm **hai lần** (lần nữa ở bước "còn lại"). Kiểm tra nhanh: độ dài kết quả phải bằng `len(arr1)`.

### Lỗi 4: Dùng `x in arr2` (list) trong khóa — đúng nhưng chậm

`x in arr2` là `O(m)`; gọi cho mỗi phần tử → `O(n·m)`. Dựng `rank` (dict) hoặc `set(arr2)` một lần rồi tra `O(1)` trung bình.

## 13. Mẹo nhận dạng pattern

- "Sắp xếp theo thứ tự cho trước / ưu tiên nhóm A rồi nhóm B" → **khóa tuple** `(nhóm, tiêu chí phụ)`.
- Thứ tự cho trước bằng một list → `rank = {v: i for i, v in enumerate(order)}`.
- Giá trị nằm trong miền nhỏ đã biết (`0..1000`, 26 chữ cái) → **counting sort** `O(n + V)`.
- Cần sắp tăng nhưng có mảng đếm → duyệt chỉ số `0..V` là ra thứ tự tăng "miễn phí".
- Bài cùng họ: #791 Custom Sort String, #1636 Sort Array by Increasing Frequency, #451 Sort Characters By Frequency, #953 Verifying an Alien Dictionary, #75 Sort Colors.

## 14. Biến thể

### 14.1. #791 Custom Sort String

Sắp các ký tự của `s` theo thứ tự xuất hiện trong `order`; ký tự không có trong `order` đặt ở đâu cũng được (ở đây: cuối, giữ thứ tự gốc).

In [22]:
def custom_sort_string(order: str, s: str) -> str:
    rank = {ch: i for i, ch in enumerate(order)}
    return "".join(sorted(s, key=lambda ch: rank.get(ch, len(order))))

print(custom_sort_string("cba", "abcd"))     # cbad
print(custom_sort_string("bcafg", "abcd"))   # bcad

cbad
bcad


### 14.2. Phần còn lại sắp **giảm dần**

Với khóa tuple, đổi `(1, x)` thành `(1, -x)` (vì giá trị là số nguyên).

In [23]:
def relative_sort_rest_desc(arr1, arr2):
    rank = {v: i for i, v in enumerate(arr2)}
    return sorted(arr1, key=lambda x: (0, rank[x]) if x in rank else (1, -x))

print(relative_sort_rest_desc([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]))   # [22, 28, 8, 6, 44, 17]

[22, 28, 8, 6, 44, 17]


## 15. Bài tập (tăng dần)

**Bài 1 — Tính tay:** `arr1 = [5, 3, 5, 1, 8, 3]`, `arr2 = [3, 5]`. Viết khóa của từng phần tử và kết quả.

**Bài 2 — Dự đoán output:** `sorted([(1, 9), (0, 9), (1, 2), (0, 1)])` và `sorted(["bb", "a", "ccc", "dd"], key=len)`.

**Bài 3 — TODO:** viết lời giải khóa tuple.

In [24]:
def exercise_relative_sort(arr1: List[int], arr2: List[int]) -> List[int]:
    # TODO 1: rank = {v: i for i, v in enumerate(arr2)}
    # TODO 2: return sorted(arr1, key=lambda x: ...)
    pass

# assert exercise_relative_sort([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]) == [22, 28, 8, 6, 17, 44]

**Bài 4 — Counting sort tổng quát:** viết lại cách 3 nhưng **không** biết trước `MAXV` (tính `max(arr1)`), và hỗ trợ giá trị âm bằng cách dịch chỉ số `x - min(arr1)`.

In [25]:
def exercise_counting_general(arr1: List[int], arr2: List[int]) -> List[int]:
    # TODO
    pass

# assert exercise_counting_general([5, -10, 3, -10], [3]) == [3, -10, -10, 5]

**Bài 5 — Sort theo tần suất (#1636):** sắp tăng theo **tần suất**; cùng tần suất thì sắp **giảm** theo giá trị.

In [26]:
def exercise_frequency_sort(nums: List[int]) -> List[int]:
    # TODO: Counter + khóa tuple
    pass

# assert exercise_frequency_sort([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2]
# assert exercise_frequency_sort([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]

**Bài 6 — Mini challenge:** kiểm tra `words` có được sắp đúng theo bảng chữ cái "ngoài hành tinh" `order` không (#953). Gợi ý: biến mỗi từ thành tuple các rank rồi so sánh tuple liền kề.

In [27]:
def exercise_is_alien_sorted(words: List[str], order: str) -> bool:
    # TODO
    pass

# assert exercise_is_alien_sorted(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz") is True
# assert exercise_is_alien_sorted(["apple", "app"], "abcdefghijklmnopqrstuvwxyz") is False

### 15.1. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `rank = {3:0, 5:1}` | 1, 8 thuộc nhóm 1 | khóa 5→(0,1), 3→(0,0), 1→(1,1), 8→(1,8) |
| 2 | so tuple từ trái | `(0,1) < (0,9) < (1,2) < (1,9)` | len ổn định: `"bb"` trước `"dd"` |
| 3 | mục 5 | `x in rank` | `(0, rank[x]) if x in rank else (1, x)` |
| 4 | `lo, hi = min, max` | `count = [0] * (hi - lo + 1)` | dùng chỉ số `x - lo` |
| 5 | `freq = Counter(nums)` | khóa `(freq[x], ...)` | `key=lambda x: (freq[x], -x)` |
| 6 | `rank` cho 26 chữ | `tuple(rank[c] for c in w)` | so `t[i] <= t[i+1]` cho mọi `i` |

## 16. Đáp án tham khảo

- **Bài 1:** khóa: 5→`(0,1)`, 3→`(0,0)`, 5→`(0,1)`, 1→`(1,1)`, 8→`(1,8)`, 3→`(0,0)` → `[3, 3, 5, 5, 1, 8]`.
- **Bài 2:** `[(0, 1), (0, 9), (1, 2), (1, 9)]` và `['a', 'bb', 'dd', 'ccc']`.

In [28]:
def answer_relative_sort(arr1, arr2):
    rank = {v: i for i, v in enumerate(arr2)}
    return sorted(arr1, key=lambda x: (0, rank[x]) if x in rank else (1, x))

def answer_counting_general(arr1, arr2):
    lo, hi = min(arr1), max(arr1)
    count = [0] * (hi - lo + 1)
    for x in arr1:
        count[x - lo] += 1
    result = []
    for v in arr2:
        result.extend([v] * count[v - lo])
        count[v - lo] = 0
    for i, c in enumerate(count):
        result.extend([i + lo] * c)
    return result

def answer_frequency_sort(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], -x))

def answer_is_alien_sorted(words, order):
    rank = {c: i for i, c in enumerate(order)}
    keys = [tuple(rank[c] for c in w) for w in words]
    return all(keys[i] <= keys[i + 1] for i in range(len(keys) - 1))

assert answer_relative_sort([5, 3, 5, 1, 8, 3], [3, 5]) == [3, 3, 5, 5, 1, 8]
assert sorted([(1, 9), (0, 9), (1, 2), (0, 1)]) == [(0, 1), (0, 9), (1, 2), (1, 9)]
assert sorted(["bb", "a", "ccc", "dd"], key=len) == ["a", "bb", "dd", "ccc"]
assert answer_relative_sort([28, 6, 22, 8, 44, 17], [22, 28, 8, 6]) == [22, 28, 8, 6, 17, 44]
assert answer_counting_general([5, -10, 3, -10], [3]) == [3, -10, -10, 5]
assert answer_frequency_sort([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2]
assert answer_frequency_sort([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]
assert answer_is_alien_sorted(["hello", "leetcode"], "hlabcdefgijkmnopqrstuvwxyz") is True
assert answer_is_alien_sorted(["apple", "app"], "abcdefghijklmnopqrstuvwxyz") is False
for _ in range(500):
    arr1 = [rng.randint(-20, 20) for _ in range(rng.randint(1, 15))]
    d = list(set(arr1)); rng.shuffle(d); arr2 = d[:rng.randint(1, len(d))]
    assert answer_counting_general(arr1, arr2) == answer_relative_sort(arr1, arr2)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


Ở bài 6, so sánh tuple `(…)` cũng xử lý đúng trường hợp **tiền tố**: `("app")` ngắn hơn `("apple")` nên nhỏ hơn — vì vậy `["apple", "app"]` không được sắp đúng.

## 17. Template nhớ nhanh

```python
rank = {v: i for i, v in enumerate(order)}
result = sorted(items, key=lambda x: (0, rank[x]) if x in rank else (1, x))
```

Counting sort khi miền giá trị nhỏ:

```python
count = [0] * (MAXV + 1)
for x in arr: count[x] += 1
for v in order: out += [v] * count[v]; count[v] = 0
for v in range(MAXV + 1): out += [v] * count[v]
```

## 18. Checklist submit

- [ ] Phần tử thuộc `arr2` theo đúng thứ tự `arr2`, giá trị lặp đứng liền nhau?
- [ ] Phần ngoài `arr2` **tăng dần** (không giữ thứ tự gốc)?
- [ ] Tra cứu thuộc `arr2` bằng dict/set, không bằng list?
- [ ] Counting sort: đã reset `count[v] = 0` sau bước `arr2`?
- [ ] Độ dài kết quả bằng `len(arr1)`?